### Perceptron entrenado para suma


In [ ]:
import sys, os  # sys: acceso al intérprete de Python; os: manejo de rutas del sistema
sys.path.append(os.path.abspath('..'))  # agrega la carpeta padre (autograd/) a las rutas donde Python busca módulos, así encuentra michigrad
from michigrad.engine import Value  # Value: número que recuerda las operaciones hechas con él y puede calcular gradientes
from michigrad.visualize import show_graph  # show_graph: dibuja el grafo de operaciones de un Value
import matplotlib.pyplot as plt  # librería para hacer gráficos
import numpy as np  # librería para trabajar con arrays y números aleatorios

In [ ]:
# Definimos las variables de entrada y salidar para generar la suma

# x: matriz de 7 filas x 2 columnas. Cada fila es un ejemplo con los dos números a sumar
x = np.array([
    [2, 3],
    [8, 8],
    [20, 25],
    [122, 127],
    [10, 15],
    [2, 2],
    [49, 51]
])

# y: matriz de 7 filas x 1 columna. Cada fila es el resultado correcto (la suma) de la fila de x
# Ej: y[0] = [5] porque x[0] = [2, 3] y 2 + 3 = 5
y = np.array([
    [5],
    [16],
    [45],
    [249],
    [25],
    [4],
    [100]
])

In [ ]:
# Ahora definimos la funcion de activacion en este caso vamos a usar la funcion RELU
# ReLU(z) = z si z > 0, y 0 si z <= 0 (deja pasar los positivos y "apaga" los negativos)
def relu(x):  # x tiene que ser un Value de michigrad
    return x.relu()  # usa la ReLU de Value, que además guarda la operación en el grafo para poder hacer backpropagation

In [ ]:
# Mostramos la grafica de la funcion de activacion
x_plot = np.arange(-10, 10, 0.1)  # valores desde -10 hasta 10 (sin incluirlo) de a 0.1 -> eje horizontal del gráfico

s = []  # acá guardamos el resultado de la ReLU para cada valor
for i in x_plot:  # recorremos cada valor del eje horizontal
    s.append(np.maximum(0, i))  # ReLU con NumPy: el mayor entre 0 e i (acá usamos números comunes, no Value)

plt.plot(x_plot, s)  # dibuja la curva: x_plot en el eje horizontal, s en el vertical

In [ ]:
# El siguiente paso es definir los pesos y el sesgo (bias) para nuestro perceptron
np.random.seed(42)  # fija la semilla: los números "aleatorios" salen iguales cada vez que se ejecuta

# Cada parámetro arranca con un valor aleatorio entre 0 y 1, envuelto en un Value para que tenga gradiente
w0 = Value(np.random.random(), name="w0")  # peso que multiplica al primer número
w1 = Value(np.random.random(), name="w1")  # peso que multiplica al segundo número
b = Value(np.random.random(), name="bias")  # sesgo: se suma al final, sin multiplicar a ninguna entrada
# El objetivo del entrenamiento es llegar a w0 ≈ 1, w1 ≈ 1, b ≈ 0  ->  1*a + 1*b + 0 = a + b


Haremos un forward pass para entender como esta el modelo y a partir de ahi ir ajustando

Ahora calcularemos el yhat para luego contrarestarlo con lo que deberia contestar

In [ ]:
yhat = []  # lista donde guardamos la predicción del modelo para cada ejemplo

for i, fila in enumerate(x):  # recorremos x: i es el número de fila (0, 1, 2...) y fila es [a, b]

    x0 = Value(float(fila[0]), name=f"x0_{i}")  # primer número de la fila, convertido a Value (ej: "x0_0")
    x1 = Value(float(fila[1]), name=f"x1_{i}")  # segundo número de la fila, convertido a Value

    prediccion = relu(x0 * w0 + x1 * w1 + b)  # la neurona: multiplica cada entrada por su peso, suma el bias y aplica ReLU

    prediccion.name = f"yhat_{i}"  # le ponemos nombre para identificarla en el grafo

    yhat.append(prediccion)  # guardamos la predicción de esta fila

In [ ]:
# Calculamos la funcion de perdida (error cuadrático medio, MSE)
errores = []  # acá guardamos el error de cada ejemplo

for i in range(len(y)):  # recorremos los 7 ejemplos (len(y) = 7)
    target = Value(float(y[i][0]), name=f"y_{i}")  # resultado correcto: y[i] es [5], y[i][0] es 5 -> Value(5.0)

    error = (yhat[i] - target) ** 2  # diferencia entre predicción y valor correcto, al cuadrado (siempre positiva)

    errores.append(error)  # guardamos el error de este ejemplo

L = sum(errores) / len(errores)  # promedio de todos los errores: un solo número que mide qué tan mal está el modelo
L.name = "L"  # nombre para el grafo

In [ ]:
# Grafo de operaciones para luego aplicar el Backpropagtion
show_graph(L)  # dibuja todas las operaciones que llevaron desde las entradas y los pesos hasta L (los gradientes todavía están en 0)

In [ ]:
L.backward()  # backpropagation: calcula para cada nodo cuánto cambia L si ese nodo cambia un poco (su gradiente, .grad)
show_graph(L)  # vuelve a dibujar el grafo, ahora con los gradientes calculados

In [ ]:
# Ahora hacemos el training loop 
for i in range(1000):  # repetimos el entrenamiento 1000 veces
    # Forward pass: calculamos las predicciones con los pesos ACTUALES (se arma un grafo nuevo en cada vuelta)
    yhat = []  # vaciamos la lista de predicciones de la vuelta anterior

    for i, fila in enumerate(x):  # recorremos cada ejemplo (este i pisa al i de afuera, pero no rompe nada porque range sigue su cuenta)

        x0 = Value(float(fila[0]), name=f"x0_{i}")  # primer número a sumar
        x1 = Value(float(fila[1]), name=f"x1_{i}")  # segundo número a sumar

        prediccion = relu(x0 * w0 + x1 * w1 + b)  # salida de la neurona con los pesos actuales

        prediccion.name = f"yhat_{i}"  # nombre para el grafo

        yhat.append(prediccion)  # guardamos la predicción
        
        errores = []  # vacía la lista de errores (como está dentro del for se ejecuta 7 veces; alcanzaría con una, fuera del for)

    # Pérdida: comparamos cada predicción con su resultado correcto
    for i in range(len(y)):  # recorremos los 7 ejemplos
        target = Value(float(y[i][0]), name=f"y_{i}")  # resultado correcto de la suma

        error = (yhat[i] - target) ** 2  # error al cuadrado de este ejemplo

        errores.append(error)  # lo guardamos

    L = sum(errores) / len(errores)  # pérdida = promedio de los errores (MSE)

        
    # Limpiamos los gradientes para la siguiente iteracion
    # (backward() SUMA a .grad, así que si no los ponemos en 0 se acumularían los de la vuelta anterior)
    w0.grad = 0
    w1.grad = 0
    b.grad = 0

    # Backward pass: calcula w0.grad, w1.grad y b.grad (hacia dónde y cuánto mover cada uno para que L suba)
    L.backward()

    # Actualizamos los pesos y el sesgo (descenso por gradiente)
    # Restamos el gradiente para ir en la dirección en que L BAJA; 0.0001 es el learning rate (tamaño del paso)
    w0.data -= 0.0001 * w0.grad
    w1.data -= 0.0001 * w1.grad
    b.data -= 0.0001 * b.grad

        
    print(L.data)  # mostramos la pérdida de esta vuelta: tendría que ir bajando

In [ ]:
relu(122 * w0 + 127 * w1 + b)  # probamos el modelo ya entrenado con 122 + 127 (debería dar cerca de 249)